In [1]:
import os
import sys
import time

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED

from topnum.regularizers import (
    FastFixPhiRegularizer, DecorrelateWithOtherPhiRegularizer, DecorrelateWithOtherPhiRegularizer2
)
from topnum.scores.intratext_coherence_score import (
    IntratextCoherenceScore,
    ComputationMethod,
    WordTopicRelatednessType,
)

In [4]:
import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

In [5]:
topicnet.__file__

! ls /home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager/

20NG.csv	 MKB10__internals	    RTL_Wiki.csv
20NG__internals  postnauka.csv		    RTL_Wiki_person.csv
api.py		 postnauka__internals	    RTL_Wiki_person__internals
Brown		 postnauka_noow.csv	    ruwiki_good__internals
Brown_BOW.csv	 postnauka_noow__internals  ruwiki_good.txt
Brown_NOOW.csv	 __pycache__		    WikiRef-220
hf		 Reuters		    wiki_ref220_bow.csv
__init__.py	 Reuters_BOW.csv	    wiki_ref220_natural_order.csv
MKB10.csv	 Reuters_NOOW.csv


In [6]:
! ls /data_mil/shared/CompressaAI/iterative/data/noow

_20_Newsgroups.csv     Post_Science_NOOW_fixed.csv
_Lenta.csv	       Post_Science_NOOW_fixed__internals
MKB_10_NOOW.csv        Post_Science_NOOW__internals
Post_Science_NOOW.csv  WikiRef_220_NOOW.csv


In [7]:
DATA_FOLDER_PATH = '/data_mil/shared/CompressaAI/iterative/data/noow'

In [8]:
! head -n 2 /data_mil/shared/CompressaAI/iterative/data/noow/Post_Science_NOOW.csv

id,raw_text,vw_text
29998.txt,материал отрицательный показатель преломление физик виктор веселаго распространение свет вещество фазовый групповой скорость метаматериалы различаться фазовый групповой скорость каков физика распространение свет вещество находить применение материал отрицательный показатель преломление рассказывать доктор физикоматематический наука виктор веселаго скорость распространяться энергия вещество обычно говорить излучение распространяться вещество со скорость n раз маленький n коэффициент преломление вещество коэффициент преломление n отношение скорость свет скорость распространение излучение вещество обычно уточняться распространяться распространение энергия распространение импульс происходить различный закон энергия распространяться со скорость называться групповой скорость много скорость свет эйнштейн сформулировать самый больший скорость излучение скорость свет кмс импульс распространяться фазовый скорость сколь угодно много скорость свет скорость входить соо

In [9]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/Post_Science_NOOW.csv',
)

dataset.get_possible_modalities()

set()

In [10]:
dataset._data.head()

,id,raw_text,vw_text
id,,,
29998.txt,29998.txt,материал отрицательный показатель преломление ...,|@word материал отрицательный показатель прело...
7770.txt,7770.txt,культурный код экономика экономист александр а...,|@word культурный код экономика экономист алек...
32230.txt,32230.txt,faq наука третий класс факт эксперимент резуль...,|@word faq наука третий класс факт эксперимент...
27293.txt,27293.txt,обрушение волна поверхность жидкость математик...,|@word обрушение волна поверхность жидкость ма...
481.txt,481.txt,существовать ли суперсимметрия мир элементарны...,|@word существовать ли суперсимметрия мир элем...


In [11]:
dataset._data['vw_text'] = dataset._data['id'] + ' ' + dataset._data['vw_text']

In [16]:
dataset._data.head()

,id,raw_text,vw_text
id,,,
29998.txt,29998.txt,материал отрицательный показатель преломление ...,29998.txt |@word материал отрицательный показа...
7770.txt,7770.txt,культурный код экономика экономист александр а...,7770.txt |@word культурный код экономика эконо...
32230.txt,32230.txt,faq наука третий класс факт эксперимент резуль...,32230.txt |@word faq наука третий класс факт э...
27293.txt,27293.txt,обрушение волна поверхность жидкость математик...,27293.txt |@word обрушение волна поверхность ж...
481.txt,481.txt,существовать ли суперсимметрия мир элементарны...,481.txt |@word существовать ли суперсимметрия ...


In [18]:
dataset._data.to_csv(f'{DATA_FOLDER_PATH}/Post_Science_NOOW_fixed.csv', index=False)

In [19]:
! head -n 2 /data_mil/shared/CompressaAI/iterative/data/noow/Post_Science_NOOW.csv

id,raw_text,vw_text
29998.txt,материал отрицательный показатель преломление физик виктор веселаго распространение свет вещество фазовый групповой скорость метаматериалы различаться фазовый групповой скорость каков физика распространение свет вещество находить применение материал отрицательный показатель преломление рассказывать доктор физикоматематический наука виктор веселаго скорость распространяться энергия вещество обычно говорить излучение распространяться вещество со скорость n раз маленький n коэффициент преломление вещество коэффициент преломление n отношение скорость свет скорость распространение излучение вещество обычно уточняться распространяться распространение энергия распространение импульс происходить различный закон энергия распространяться со скорость называться групповой скорость много скорость свет эйнштейн сформулировать самый больший скорость излучение скорость свет кмс импульс распространяться фазовый скорость сколь угодно много скорость свет скорость входить соо

In [20]:
! head -n 2 /data_mil/shared/CompressaAI/iterative/data/noow/Post_Science_NOOW_fixed.csv

id,raw_text,vw_text
29998.txt,материал отрицательный показатель преломление физик виктор веселаго распространение свет вещество фазовый групповой скорость метаматериалы различаться фазовый групповой скорость каков физика распространение свет вещество находить применение материал отрицательный показатель преломление рассказывать доктор физикоматематический наука виктор веселаго скорость распространяться энергия вещество обычно говорить излучение распространяться вещество со скорость n раз маленький n коэффициент преломление вещество коэффициент преломление n отношение скорость свет скорость распространение излучение вещество обычно уточняться распространяться распространение энергия распространение импульс происходить различный закон энергия распространяться со скорость называться групповой скорость много скорость свет эйнштейн сформулировать самый больший скорость излучение скорость свет кмс импульс распространяться фазовый скорость сколь угодно много скорость свет скорость входить соо

In [9]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/Post_Science_NOOW_fixed.csv',
)

dataset.get_possible_modalities()

{'@word'}

In [10]:
MAIN_MODALITY = '@word'

In [11]:
dataset._data.head()

,id,raw_text,vw_text
id,,,
29998.txt,29998.txt,материал отрицательный показатель преломление ...,29998.txt |@word материал отрицательный показа...
7770.txt,7770.txt,культурный код экономика экономист александр а...,7770.txt |@word культурный код экономика эконо...
32230.txt,32230.txt,faq наука третий класс факт эксперимент резуль...,32230.txt |@word faq наука третий класс факт э...
27293.txt,27293.txt,обрушение волна поверхность жидкость математик...,27293.txt |@word обрушение волна поверхность ж...
481.txt,481.txt,существовать ли суперсимметрия мир элементарны...,481.txt |@word существовать ли суперсимметрия ...


In [12]:
dataset.get_dictionary()

artm.Dictionary(name=84e8ffe7-f675-4064-99d7-f1815cbd1945, num_entries=82162)

In [13]:
dataset._data.shape

(3446, 3)

In [14]:
dictionary = dataset.get_dictionary()

print(dictionary)

for modality in dataset.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=84e8ffe7-f675-4064-99d7-f1815cbd1945, num_entries=82162)


In [15]:
dictionary

artm.Dictionary(name=84e8ffe7-f675-4064-99d7-f1815cbd1945, num_entries=82162)

In [16]:
dataset._data.shape

(3446, 3)

In [17]:
dictionary.filter(min_df=5, max_df_rate=0.5)

artm.Dictionary(name=84e8ffe7-f675-4064-99d7-f1815cbd1945, num_entries=19537)

In [18]:
dataset._cached_dict = dictionary

In [19]:
dataset.get_dictionary()

artm.Dictionary(name=84e8ffe7-f675-4064-99d7-f1815cbd1945, num_entries=19537)

In [20]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))  # TODO: sizes * (sizes - 1) ? no need for multiplier 2 ?
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [21]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 6.05 s, sys: 251 ms, total: 6.3 s
Wall time: 6.22 s


In [22]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [23]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [24]:
KnownModel

<enum 'KnownModel'>

In [25]:
PARAMS_EXPLORED

{<KnownModel.LDA: 'LDA'>: {'prior': ['symmetric', 'asymmetric', 'heuristic']},
 <KnownModel.PLSA: 'PLSA'>: {},
 <KnownModel.TLESS: 'TARTM'>: {},
 <KnownModel.SPARSE: 'sparse'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1]},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': [0.02,
   0.05,
   0.1]},
 <KnownModel.ARTM: 'ARTM'>: {'smooth_bcg_tau': [0.05, 0.1],
  'sparse_sp_tau': [-0.05, -0.1],
  'decorrelation_tau': [0.02, 0.05, 0.1]}}

In [26]:
NUM_TOPICS = 20  # vary
NUM_TRAINS = 3
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

## Test

In [27]:
PARAMS_EXPLORED[KnownModel.PLSA]


model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=1,
)

model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [28]:
model.get_phi().shape

(19537, 20)

In [29]:
list(model.scores.keys())

['PerplexityScore@all',
 'SparsityThetaScore',
 'SparsityPhiScore@word',
 'PerplexityScore@word',
 'TopicKernel@word.average_coherence',
 'TopicKernel@word.average_contrast',
 'TopicKernel@word.average_purity',
 'TopicKernel@word.average_size',
 'TopicKernel@word.coherence',
 'TopicKernel@word.contrast',
 'TopicKernel@word.purity',
 'TopicKernel@word.size',
 'TopicKernel@word.tokens']

In [30]:
model.scores['PerplexityScore@word']

[19201.404296875,
 4362.3388671875,
 4016.84814453125,
 3453.7880859375,
 3084.326171875,
 2892.353759765625,
 2776.40869140625,
 2699.38134765625,
 2645.42724609375,
 2606.574951171875,
 2578.23681640625,
 2557.467529296875,
 2542.052490234375,
 2530.478515625,
 2521.54443359375,
 2514.327392578125,
 2508.275146484375,
 2503.078125,
 2498.58056640625,
 2494.68310546875]

In [ ]:
phi = model.get_phi()
target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
target_topic_names = [phi.columns[i] for i in target_topic_indices]

custom_scores = [
    TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )
    for top in [20]  # [10, 20, 50, 100]
]
custom_scores = custom_scores + [
    DiversityScore(
        name=f'diversity_{metric}',
        topic_names=['topic_0', 'topic_1'],
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

for score in custom_scores:    
    res = score.call(model)

    print(score._name)
    print(res)

    if isinstance(score, TopTokenCoherence):
        res_by_topic = score.call_by_topic(model)

        print(res_by_topic)

    print()

In [31]:
for score in custom_scores:
    if isinstance(score, TopTokenCoherence):
        break

toptok = score

In [118]:
toptok

TopTokenCoherence

In [121]:
%%timeit

res = toptok.call(model)

72.8 ms ± 21.1 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [120]:
intra1 = IntratextCoherenceScore(
    name='toplen',
    data=dataset,
    documents=list(dataset._data.index)[:1],
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
)
intra10 = IntratextCoherenceScore(
    name='toplen',
    data=dataset,
    documents=list(dataset._data.index)[:10],
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
)
intra100 = IntratextCoherenceScore(
    name='toplen',
    data=dataset,
    documents=list(dataset._data.index)[:100],
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
)

In [122]:
%%timeit

res = intra1.call(model)

271 ms ± 581 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [123]:
%%timeit

res = intra10.call(model)

2.43 s ± 6.55 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [124]:
%%time

res = intra100.call(model)

CPU times: user 26.4 s, sys: 0 ns, total: 26.4 s
Wall time: 25.3 s


In [103]:
intra_scores = [
    IntratextCoherenceScore(
        name='toplen',
        data=dataset,
        documents=list(dataset._data.index)[:1],
        computation_method=ComputationMethod.SEGMENT_LENGTH,
        word_topic_relatedness=WordTopicRelatednessType.PTW,
    ),
    IntratextCoherenceScore(
        name='toplen2',
        data=dataset,
        computation_method=ComputationMethod.SUM_OVER_WINDOW,
        word_topic_relatedness=WordTopicRelatednessType.PTW,
    ),
]

In [104]:
%%time

intra_scores[0].call(model)

CPU times: user 328 ms, sys: 6.14 ms, total: 334 ms
Wall time: 314 ms


1.3095238095238095

In [70]:
%%time

intra_scores[1].call(model)

KeyboardInterrupt: 

In [71]:
_phi = model.get_phi()

In [72]:
_phi.head()

topic_0  topic_1  ...      topic_18      topic_19
modality token                            ...                            
@word    дублирование  0.000000      0.0  ...  0.000000e+00  2.014671e-05
         проучиться    0.000014      0.0  ...  7.175488e-11  7.105283e-09
         гимнастика    0.000000      0.0  ...  6.034839e-06  0.000000e+00
         софист        0.000000      0.0  ...  0.000000e+00  0.000000e+00
         парижанин     0.000000      0.0  ...  0.000000e+00  0.000000e+00

[5 rows x 20 columns]

In [76]:
intra_scores[0]._score._get_relatedness(
    ('@word', 'проучиться'), 'topic_0', _phi
)

1.4186415e-05

In [79]:
words = dataset._data.iloc[0, 2].split()[2:]

In [80]:
words

['материал',
 'отрицательный',
 'показатель',
 'преломление',
 'физик',
 'виктор',
 'веселаго',
 'распространение',
 'свет',
 'вещество',
 'фазовый',
 'групповой',
 'скорость',
 'метаматериалы',
 'различаться',
 'фазовый',
 'групповой',
 'скорость',
 'каков',
 'физика',
 'распространение',
 'свет',
 'вещество',
 'находить',
 'применение',
 'материал',
 'отрицательный',
 'показатель',
 'преломление',
 'рассказывать',
 'доктор',
 'физикоматематический',
 'наука',
 'виктор',
 'веселаго',
 'скорость',
 'распространяться',
 'энергия',
 'вещество',
 'обычно',
 'говорить',
 'излучение',
 'распространяться',
 'вещество',
 'со',
 'скорость',
 'n',
 'раз',
 'маленький',
 'n',
 'коэффициент',
 'преломление',
 'вещество',
 'коэффициент',
 'преломление',
 'n',
 'отношение',
 'скорость',
 'свет',
 'скорость',
 'распространение',
 'излучение',
 'вещество',
 'обычно',
 'уточняться',
 'распространяться',
 'распространение',
 'энергия',
 'распространение',
 'импульс',
 'происходить',
 'различный',
 'зак

In [109]:
def _get_relatedness(
        word,
        topic: str,
        word_topic_relatednesses) -> float:

    try:
        return 0  # word_topic_relatednesses.loc[word, topic]
    except KeyError as error:
        # print(
        #     f'Some word not found in Word-Topic relatedness matrix: "{error}"!'
        #     f' Returning mean value over all word relatednesses for topic "{topic}".'
        # )

        return 0  # float(np.mean(word_topic_relatednesses.values))

In [110]:
%%time

for i in range(1):
    for t in _phi.columns:
        for w in words:
            _get_relatedness(
                ('@word', w), t, _phi
            )

CPU times: user 2.18 ms, sys: 0 ns, total: 2.18 ms
Wall time: 2 ms


In [112]:
_phi.index[0]

('@word', 'дублирование')

In [113]:
_phi.to_dict()

{'topic_0': {('@word', 'дублирование'): 0.0,
  ('@word', 'проучиться'): 1.4186414773575962e-05,
  ('@word', 'гимнастика'): 0.0,
  ('@word', 'софист'): 0.0,
  ('@word', 'парижанин'): 0.0,
  ('@word', 'жадный'): 0.0,
  ('@word', 'антиd'): 8.654332486912608e-05,
  ('@word', 'volk'): 0.0,
  ('@word', 'огонек'): 0.0,
  ('@word', 'несторианский'): 0.0,
  ('@word', 'грайндхаус'): 0.0,
  ('@word', 'будить'): 0.0,
  ('@word', 'кихот'): 0.0,
  ('@word', 'алетейя'): 0.0,
  ('@word', 'халат'): 0.0,
  ('@word', 'винительный'): 0.0,
  ('@word', 'истерический'): 0.0,
  ('@word', 'якобинский'): 0.0,
  ('@word', 'шестеро'): 0.0,
  ('@word', 'уйгур'): 0.0,
  ('@word', 'фарси'): 0.0,
  ('@word', 'биосовместимый'): 0.0,
  ('@word', 'впрыскивать'): 1.5044696738186758e-05,
  ('@word', 'зарастать'): 0.0,
  ('@word', 'канава'): 0.0,
  ('@word', 'желудочек'): 0.0,
  ('@word', 'поручаться'): 0.0,
  ('@word', 'ja'): 0.0,
  ('@word', 'портовый'): 0.0,
  ('@word', 'придумывание'): 0.0,
  ('@word', 'таксист'): 0.0,

In [125]:
np.mean(_phi)

5.1185107e-05

In [32]:
# With _word_topic_relatednesses_fast:

In [33]:
intra1 = IntratextCoherenceScore(
    name='toplen',
    data=dataset,
    documents=list(dataset._data.index)[:1],
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
)
intra10 = IntratextCoherenceScore(
    name='toplen',
    data=dataset,
    documents=list(dataset._data.index)[:10],
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
)
intra100 = IntratextCoherenceScore(
    name='toplen',
    data=dataset,
    documents=list(dataset._data.index)[:100],
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
)

In [34]:
%%timeit

res = intra1.call(model)

241 ms ± 2.83 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [35]:
%%timeit

res = intra10.call(model)

467 ms ± 5.55 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [36]:
%%time

res = intra100.call(model)

CPU times: user 3.02 s, sys: 29.4 ms, total: 3.05 s
Wall time: 2.9 s


In [37]:
intra = IntratextCoherenceScore(
    name='toplen',
    data=dataset,
    documents=list(dataset._data.index),
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
)

In [38]:
%%time

res = intra.call(model)

CPU times: user 1min 40s, sys: 1.58 s, total: 1min 41s
Wall time: 1min 36s


In [39]:
NUM_ITERATIONS

20

In [40]:
%%timeit

model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

21.2 s ± 245 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
intra1 = IntratextCoherenceScore(
    name='toplen_pwt',
    data=dataset,
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PWT,
    should_compute=False,  # only on last iter
)
intra2 = IntratextCoherenceScore(
    name='toplen_ptw',
    data=dataset,
    computation_method=ComputationMethod.SEGMENT_LENGTH,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
    should_compute=False,
)
intra3 = IntratextCoherenceScore(
    name='topden_ptw',
    data=dataset,
    computation_method=ComputationMethod.SUM_OVER_WINDOW,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
    should_compute=False,
)

###

intra3_w4 = IntratextCoherenceScore(
    name='topden_ptw_w4',
    data=dataset,
    computation_method=ComputationMethod.SUM_OVER_WINDOW,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
    window=4,
    should_compute=False,
)
intra3_w2 = IntratextCoherenceScore(
    name='topden_ptw_w2',
    data=dataset,
    computation_method=ComputationMethod.SUM_OVER_WINDOW,
    word_topic_relatedness=WordTopicRelatednessType.PTW,
    window=2,
    should_compute=False,
)

In [37]:
%%time

res = intra1.call(model)

CPU times: user 1min 39s, sys: 1.38 s, total: 1min 40s
Wall time: 1min 35s


In [38]:
%%time

res = intra2.call(model)

CPU times: user 1min 38s, sys: 1.61 s, total: 1min 40s
Wall time: 1min 35s


In [39]:
%%time

res = intra3.call(model)

CPU times: user 2min 36s, sys: 5.96 s, total: 2min 42s
Wall time: 2min 36s


In [94]:
%%time

res = intra3_w4.call(model)

CPU times: user 2min 21s, sys: 4.61 s, total: 2min 26s
Wall time: 2min 16s


In [95]:
%%time

res = intra3_w2.call(model)

CPU times: user 2min 12s, sys: 3.5 s, total: 2min 15s
Wall time: 2min 6s


In [ ]:
# After try-catch in _get_word_topic_index instead of if-else:

In [30]:
%%time

res = intra1.call(model)

CPU times: user 1min 36s, sys: 1.53 s, total: 1min 38s
Wall time: 1min 33s


In [31]:
%%time

res = intra2.call(model)

CPU times: user 1min 36s, sys: 1.95 s, total: 1min 38s
Wall time: 1min 33s


In [32]:
%%time

res = intra3.call(model)

CPU times: user 2min 34s, sys: 5.95 s, total: 2min 40s
Wall time: 2min 33s


In [33]:
%%time

res = intra3_w4.call(model)

CPU times: user 2min 16s, sys: 3.46 s, total: 2min 20s
Wall time: 2min 14s


In [34]:
%%time

res = intra3_w2.call(model)

CPU times: user 2min 6s, sys: 2.91 s, total: 2min 9s
Wall time: 2min 4s


In [ ]:
# After removing np.floor from self._window // 2

In [29]:
%%time

res = intra3.call(model)

CPU times: user 2min 28s, sys: 5.52 s, total: 2min 34s
Wall time: 2min 27s


In [30]:
%%time

res = intra3_w4.call(model)

CPU times: user 2min 9s, sys: 3.62 s, total: 2min 13s
Wall time: 2min 7s


In [31]:
%%time

res = intra3_w2.call(model)

CPU times: user 2min 1s, sys: 2.33 s, total: 2min 3s
Wall time: 1min 58s


In [ ]:
# After careful handling of left border (so as to eliminate intersections)

In [33]:
%%time

res = intra3.call(model)

CPU times: user 2min 25s, sys: 5.46 s, total: 2min 30s
Wall time: 2min 24s


In [34]:
%%time

res = intra3_w4.call(model)

CPU times: user 2min 9s, sys: 3.32 s, total: 2min 12s
Wall time: 2min 7s


In [35]:
%%time

res = intra3_w2.call(model)

CPU times: user 2min 1s, sys: 2.63 s, total: 2min 3s
Wall time: 1min 58s


In [28]:
# After np.sum -> sum

In [29]:
%%time

res = intra3.call(model)

CPU times: user 2min 12s, sys: 4.59 s, total: 2min 17s
Wall time: 2min 11s


In [30]:
%%time

res = intra3_w4.call(model)

CPU times: user 1min 58s, sys: 2.81 s, total: 2min
Wall time: 1min 55s


In [31]:
%%time

res = intra3_w2.call(model)

CPU times: user 1min 50s, sys: 2.2 s, total: 1min 52s
Wall time: 1min 47s


In [ ]:
# After sum(list) -> v + dv

In [32]:
%%time

res = intra3.call(model)

CPU times: user 2min 12s, sys: 4.73 s, total: 2min 17s
Wall time: 2min 11s


In [33]:
%%time

res = intra3_w4.call(model)

CPU times: user 1min 57s, sys: 2.76 s, total: 2min
Wall time: 1min 55s


In [34]:
%%time

res = intra3_w2.call(model)

CPU times: user 1min 49s, sys: 2.21 s, total: 1min 51s
Wall time: 1min 47s


In [ ]:
# After lru cache (unlimited) for _get_relatedness

In [34]:
%%time

res = intra3.call(model)

CPU times: user 1min 56s, sys: 2.66 s, total: 1min 58s
Wall time: 1min 54s


In [35]:
%%time

res = intra3_w4.call(model)

CPU times: user 1min 50s, sys: 2.09 s, total: 1min 52s
Wall time: 1min 47s


In [36]:
%%time

res = intra3_w2.call(model)

CPU times: user 1min 45s, sys: 1.55 s, total: 1min 47s
Wall time: 1min 43s


In [ ]:
# After lru cache (5_000_000) for _get_relatedness

In [32]:
%%time

res = intra3.call(model)

CPU times: user 1min 58s, sys: 2.67 s, total: 2min 1s
Wall time: 1min 56s


In [33]:
%%time

res = intra3_w4.call(model)

CPU times: user 1min 51s, sys: 2.26 s, total: 1min 53s
Wall time: 1min 49s


In [34]:
%%time

res = intra3_w2.call(model)

CPU times: user 1min 47s, sys: 1.5 s, total: 1min 48s
Wall time: 1min 44s


In [ ]:
# After lru cache (unlimited) for _get_relatedness and _get_word_topic_index

In [34]:
%%time

res = intra2.call(model)

CPU times: user 1min 39s, sys: 2.01 s, total: 1min 41s
Wall time: 1min 36s


In [31]:
%%time

res = intra3.call(model)

CPU times: user 1min 59s, sys: 2.78 s, total: 2min 2s
Wall time: 1min 57s


In [32]:
%%time

res = intra3_w4.call(model)

CPU times: user 1min 53s, sys: 1.87 s, total: 1min 55s
Wall time: 1min 50s


In [33]:
%%time

res = intra3_w2.call(model)

CPU times: user 1min 49s, sys: 1.43 s, total: 1min 51s
Wall time: 1min 46s


In [ ]:
# lru cache (unlimited) for _get_relatedness only

In [33]:
%%time

res = intra2.call(model)

CPU times: user 1min 36s, sys: 2.07 s, total: 1min 38s
Wall time: 1min 33s


In [34]:
%%time

res = intra3.call(model)

CPU times: user 1min 57s, sys: 2.64 s, total: 2min
Wall time: 1min 55s


In [35]:
%%time

res = intra3_w4.call(model)

CPU times: user 1min 51s, sys: 1.72 s, total: 1min 52s
Wall time: 1min 48s


In [36]:
%%time

res = intra3_w2.call(model)

CPU times: user 1min 45s, sys: 1.84 s, total: 1min 47s
Wall time: 1min 42s


In [37]:
model.get_phi(class_ids=MAIN_MODALITY)['topic_18'].sort_values(ascending=False)

modality  token            
@word     система              0.008737
          технология           0.007684
          задача               0.007392
          данные               0.006979
          сеть                 0.005978
                                 ...   
          ограда               0.000000
          насильственно        0.000000
          ячмень               0.000000
          тераэлектронвольт    0.000000
          дублирование         0.000000
Name: topic_18, Length: 19537, dtype: float32

In [38]:
model.class_ids

{'@word': 1}

In [39]:
KNOWN_METRICS

['euclidean', 'jensenshannon', 'hellinger', 'cosine']

In [40]:
MAIN_MODALITY

'@word'

In [41]:
def fit_and_compute_scores(model, dataset):
    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    
    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    # print(f'Computing "{coherence_score._name}"...')
    
    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    # print(f'Result by topic: {topic_coherences}.')


    # intra1 = IntratextCoherenceScore(
    #     name='toplen_pwt',
    #     data=dataset,
    #     computation_method=ComputationMethod.SEGMENT_LENGTH,
    #     word_topic_relatedness=WordTopicRelatednessType.PWT,
    #     should_compute=False,  # only on last iter
    # )
    intra2 = IntratextCoherenceScore(
        name='toplen_ptw',
        data=dataset,
        computation_method=ComputationMethod.SEGMENT_LENGTH,
        word_topic_relatedness=WordTopicRelatednessType.PTW,
        should_compute=False,
    )
    # intra3 = IntratextCoherenceScore(
    #     name='topden_ptw',
    #     data=dataset,
    #     computation_method=ComputationMethod.SUM_OVER_WINDOW,
    #     word_topic_relatedness=WordTopicRelatednessType.PTW,
    #     should_compute=False,
    # )
    intra3_w4 = IntratextCoherenceScore(
        name='topden_ptw',
        data=dataset,
        computation_method=ComputationMethod.SUM_OVER_WINDOW,
        word_topic_relatedness=WordTopicRelatednessType.PTW,
        window=4,
        should_compute=False,
    )

    intra_topic_coherences = dict()

    for intra in [intra2, intra3_w4]:  #[intra1, intra2, intra3]:
        # print(f'\nComputing "{intra._name}"...')

        current_intra_topic_coherences = intra.compute(model)

        assert all(v is not None for v in current_intra_topic_coherences.values())

        _values = current_intra_topic_coherences.values()

        current_intra_topic_coherences = {
            i: current_intra_topic_coherences[t]  # if v is not None else 0.0
            for i, t in enumerate(target_topic_names)
        }

        assert all(abs(x - y) <= 1e-6 for x, y in zip(_values, current_intra_topic_coherences.values())), (_values, current_intra_topic_coherences.values())  # "sorted" Python dicts
        
        intra_topic_coherences[f'topic_coherences_{intra._name}'] = current_intra_topic_coherences

        value = float(np.median(list(current_intra_topic_coherences.values())))
        score_values[intra._name] = value

        # print(f'Result by topic: {current_intra_topic_coherences}.')


    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    
    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
        **intra_topic_coherences,
    }

In [42]:
BEST_PARAMS = dict()

## PLSA

In [43]:
PARAMS_EXPLORED[KnownModel.PLSA]

{}

In [44]:
NUM_TOPICS

20

In [45]:
model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=0,
)

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [71]:
start = time.time()

scores = fit_and_compute_scores(model, dataset)

end = time.time()

print(f'Elapsed: {(end - start) / 60} min')

Computing "coherence_20"...
Result by topic: {0: 0.4682133200037822, 1: 0.5525499039748306, 2: 0.3662412591097539, 3: 0.7660746688638103, 4: 0.6599584504901529, 5: 1.2151843412918593, 6: 0.784374938331295, 7: 0.6583524284174844, 8: 0.7558873960285204, 9: 0.8872288724338003, 10: 0.5060579830596448, 11: 1.1643195534386466, 12: 1.0551562205101295, 13: 0.6667121824386473, 14: 0.34910355580365166, 15: 0.7823822821267535, 16: 1.0022462210267367, 17: 0.8400496475709041, 18: 0.706545516531718, 19: 0.425808078140945}.

Computing "toplen_pwt"...
Result by topic: {0: 2.0956432149414908, 1: 2.1535240947120737, 2: 2.8292371579349185, 3: 1.87571387898929, 4: 2.346451870009993, 5: 3.226761710980802, 6: 1.9268920456173568, 7: 1.6180633470149492, 8: 2.1603641420965194, 9: 2.111364111726634, 10: 1.8843975960799404, 11: 2.67100155859519, 12: 2.0041191448861375, 13: 2.2960403613814058, 14: 2.29035348424422, 15: 2.2511018392627333, 16: 3.2202474401611747, 17: 2.1978257467664424, 18: 2.246628765802841, 19: 

In [41]:
start = time.time()

scores = fit_and_compute_scores(model, dataset)

end = time.time()

print(f'\nElapsed: {(end - start) / 60} min')

Elapsed: 4.016988714536031 min


In [46]:
start = time.time()

scores = fit_and_compute_scores(model, dataset)

end = time.time()

print(f'\nElapsed: {(end - start) / 60} min')


Elapsed: 3.7278953472773235 min


In [47]:
scores

{'scores': {'perplexity': 2474.488525390625,
  'coherence_20': array([0.73062234]),
  'toplen_ptw': 2.2222272562846417,
  'topden_ptw': 0.4862072399044338,
  'diversity_euclidean': 0.05432938513130658,
  'diversity_jensenshannon': 0.6000053851445594,
  'diversity_hellinger': 0.6857673704489131,
  'diversity_cosine': 0.7377318942011689},
 'topic_coherences': {0: 0.4682133200037822,
  1: 0.5525499039748306,
  2: 0.3662412591097539,
  3: 0.7660746688638103,
  4: 0.6599584504901529,
  5: 1.2151843412918593,
  6: 0.784374938331295,
  7: 0.6583524284174844,
  8: 0.7558873960285204,
  9: 0.8872288724338003,
  10: 0.5060579830596448,
  11: 1.1643195534386466,
  12: 1.0551562205101295,
  13: 0.6667121824386473,
  14: 0.34910355580365166,
  15: 0.7823822821267535,
  16: 1.0022462210267367,
  17: 0.8400496475709041,
  18: 0.706545516531718,
  19: 0.425808078140945},
 'topic_coherences_toplen_ptw': {0: 2.0956432149414908,
  1: 2.1535240947120737,
  2: 2.8292371579349185,
  3: 1.87571387898929,
  4

In [26]:
BEST_PARAMS[KnownModel.PLSA] = None

## Sparse

In [27]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [39]:
results = dict()

for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['sparse_sp_tau']:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (sparse_sp_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.SPARSE,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={'sparse_sp_tau': sparse_sp_tau, 'smooth_bcg_tau': smooth_bcg_tau}
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
 
            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(-0.05, 0.05)
0
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027

(-0.05, 0.1)
0


...

smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314




In [41]:
results

{(-0.05,
  0.05): [{'scores': {'perplexity': 3350.7021484375,
    'coherence_20': array([0.79300455]),
    'diversity_euclidean': 0.06043655558783355,
    'diversity_jensenshannon': 0.06043655558783355,
    'diversity_hellinger': 0.06043655558783355,
    'diversity_cosine': 0.06043655558783355},
   'topic_coherences': {0: 0.4613815887605477,
    1: 0.7254412703733861,
    2: 1.1825977284049987,
    3: 0.8628769522261815,
    4: 0.9995690399053021,
    5: 0.9733058107435728,
    6: 0.6980303292285863,
    7: 0.5761489198688887,
    8: 0.7483147931845104,
    9: 0.8621148798441607,
    10: 1.1127012218906336,
    11: 0.5671635792800231,
    12: 0.7538572393166724,
    13: 0.5220896659030285,
    14: 0.5394795375149565,
    15: 0.8817214328684023,
    16: 1.1619803113527283,
    17: 0.6217772221062762,
    18: 0.8775876214854835,
    19: 0.7319519205247833}}, {'scores': {'perplexity': 3328.0341796875,
    'coherence_20': array([0.78209502]),
    'diversity_euclidean': 0.05763263478401777,

In [42]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

(-0.05, 0.05) 3331.8850911458335
(-0.05, 0.1) 3491.5355631510415
(-0.1, 0.05) 3475.8790690104165
(-0.1, 0.1) 3637.692138671875


In [ ]:
# Best: (-0.05, 0.05) 3331.8850911458335

In [27]:
BEST_PARAMS[KnownModel.SPARSE] = {
    'sparse_sp_tau': -0.05,
    'smooth_bcg_tau': 0.05,
}

## Decorrelation

In [17]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [23]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [32]:
DECORRELATION_TAUS = [0.01] + PARAMS_EXPLORED[KnownModel.DECORRELATION]['decorrelation_tau']

In [33]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
        key = (decorrelation_tau, smooth_bcg_tau)
        results[key] = []

        print(key)

        for seed in range(NUM_TRAINS):
            print(seed)
            
            model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': decorrelation_tau,
                    'smooth_bcg_tau': smooth_bcg_tau,
                    'sparse_sp_tau': 0.0,
                }
            )

            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")

            scores = fit_and_compute_scores(model, dataset)
            results[key].append(scores)

        print()

    print()

(0.02, 0.05)
0
smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.02
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.02
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.02

(0.02, 0.1)
0


...

smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: 0.0
sparse_theta_sp: 0.0
decorrelation: 0.01




In [34]:
len(results)

8

In [35]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    print(k, mean_ppl)

(0.02, 0.05) 3080.7899576822915
(0.02, 0.1) 3209.5741373697915
(0.05, 0.05) 3085.8776041666665
(0.05, 0.1) 3216.2530110677085
(0.1, 0.05) 3148.940673828125
(0.1, 0.1) 3275.7158203125
(0.01, 0.05) 3081.0397135416665
(0.01, 0.1) 3209.85693359375


In [ ]:
# Best: (0.02, 0.05) 3080.7899576822915

In [19]:
BEST_PARAMS[KnownModel.DECORRELATION] = {
    'decorrelation_tau': 0.02,
    'smooth_bcg_tau': 0.05,
}

## ARTM

In [27]:
PARAMS_EXPLORED[KnownModel.SPARSE]

{'smooth_bcg_tau': [0.05, 0.1], 'sparse_sp_tau': [-0.05, -0.1]}

In [28]:
PARAMS_EXPLORED[KnownModel.DECORRELATION]

{'decorrelation_tau': [0.02, 0.05, 0.1]}

In [29]:
PARAMS_EXPLORED[KnownModel.ARTM]

{'smooth_bcg_tau': [0.05, 0.1],
 'sparse_sp_tau': [-0.05, -0.1],
 'decorrelation_tau': [0.02, 0.05, 0.1]}

In [36]:
results = dict()

for decorrelation_tau in DECORRELATION_TAUS:
    for sparse_sp_tau in PARAMS_EXPLORED[KnownModel.ARTM]['sparse_sp_tau']:
        for smooth_bcg_tau in PARAMS_EXPLORED[KnownModel.SPARSE]['smooth_bcg_tau']:
            key = (decorrelation_tau, sparse_sp_tau, smooth_bcg_tau)
            results[key] = []
    
            print(key)
    
            for seed in range(NUM_TRAINS):
                print(seed)
                
                model = init_model_from_family(
                    family=KnownModel.ARTM,
                    dataset=dataset,
                    main_modality=MAIN_MODALITY,
                    num_topics=NUM_TOPICS,
                    seed=seed,
                    model_params={
                        'decorrelation_tau': decorrelation_tau,
                        'smooth_bcg_tau': smooth_bcg_tau,
                        'sparse_sp_tau': sparse_sp_tau,
                    }
                )
    
                for reg in model.regularizers.data:
                    print(f"{reg}: {model.regularizers[reg].tau}")
    
                scores = fit_and_compute_scores(model, dataset)
                results[key].append(scores)

            print()

        print()

    print()

(0.02, -0.05, 0.05)
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
decorrelation: 0.02
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 3.9330432826567625
smooth_theta_bcg: 22.167852062588906
sparse_phi_sp: -0.17792338659637733
sparse_theta_sp: -1.0028314028314027
decorrelation: 0.02
2


...

smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314
decorrelation: 0.01
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314
decorrelation: 0.01
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi_bcg: 8.30309137449761
smooth_theta_bcg: 46.7987987987988
sparse_phi_sp: -0.33967191986581124
sparse_theta_sp: -1.914496314496314
decorrelation: 0.01





In [37]:
len(results)

16

In [38]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

(0.02, -0.05, 0.05) 3341.3675130208335
(0.02, -0.05, 0.1) 3497.8352864583335
(0.02, -0.1, 0.05) 3486.6405436197915
(0.02, -0.1, 0.1) 3645.3518880208335
(0.05, -0.05, 0.05) 3367.4226888020835
(0.05, -0.05, 0.1) 3518.19677734375
(0.05, -0.1, 0.05) 3512.6874186197915
(0.05, -0.1, 0.1) 3666.226318359375
(0.1, -0.05, 0.05) 3442.3375651041665
(0.1, -0.05, 0.1) 3582.7361653645835
(0.1, -0.1, 0.05) 3589.3958333333335
(0.1, -0.1, 0.1) 3733.7527669270835
(0.01, -0.05, 0.05) 3335.7916666666665
(0.01, -0.05, 0.1) 3493.9044596354165
(0.01, -0.1, 0.05) 3480.2273763020835
(0.01, -0.1, 0.1) 3640.7910970052085


In [ ]:
# Best: (0.01, -0.05, 0.05) 3335.7916666666665

# Close: (0.02, -0.05, 0.05) 3341.3675130208335

In [20]:
# BEST_PARAMS[KnownModel.DECORRELATION] = {
#     'decorrelation_tau': 0.01,
#     'sparse_sp_tau': -0.05,
#     'smooth_bcg_tau': 0.05,
# }

BEST_PARAMS[KnownModel.ARTM] = {
    'decorrelation_tau': 0.01,
    'sparse_sp_tau': -0.05,
    'smooth_bcg_tau': 0.05,
}

## TLESS

In [19]:
PARAMS_EXPLORED[KnownModel.TLESS]

{}

In [20]:
results = []

for seed in range(NUM_TRAINS):
    print(seed)
    
    model = init_model_from_family(
        family=KnownModel.TLESS,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    scores = fit_and_compute_scores(model, dataset)
    results.append(scores)

0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


In [22]:
results

[{'scores': {'perplexity': 3648.019287109375,
   'coherence_20': array([0.73268797]),
   'diversity_euclidean': 0.0828906236701155,
   'diversity_jensenshannon': 0.0828906236701155,
   'diversity_hellinger': 0.0828906236701155,
   'diversity_cosine': 0.0828906236701155},
  'topic_coherences': {0: 0.5748634448468164,
   1: 0.730467322022561,
   2: 0.4112179157799582,
   3: 1.008451239401693,
   4: 1.1553236136598417,
   5: 0.5280133176474108,
   6: 0.597173470430456,
   7: 0.4813047901914537,
   8: 0.8452729644551105,
   9: 1.1083068584696496,
   10: 1.0975959440604885,
   11: 0.5723046051892471,
   12: 0.8491850406145895,
   13: 0.5719212219989672,
   14: 0.4161534136724659,
   15: 0.9974945274670723,
   16: 1.3190569328602562,
   17: 0.4682329257973112,
   18: 0.5698427247201727,
   19: 0.35157720925244956}},
 {'scores': {'perplexity': 3635.931884765625,
   'coherence_20': array([0.71696431]),
   'diversity_euclidean': 0.08295491382261991,
   'diversity_jensenshannon': 0.0829549138226

In [ ]:
# Best:

In [30]:
BEST_PARAMS[KnownModel.TLESS] = None

## LDA

In [41]:
PARAMS_EXPLORED[KnownModel.LDA]

{'prior': ['symmetric', 'asymmetric', 'heuristic']}

In [42]:
results = dict()

for prior in PARAMS_EXPLORED[KnownModel.LDA]['prior']:
    key = prior
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.LDA,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={'prior': prior}
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        scores = fit_and_compute_scores(model, dataset)
        results[key].append(scores)

    print()

symmetric
0


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.05
smooth_theta: 0.05
2


...

smooth_phi: 0.01
smooth_theta: 2.5
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5
2


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


smooth_phi: 0.01
smooth_theta: 2.5



In [43]:
results

{'symmetric': [{'scores': {'perplexity': 2994.8544921875,
    'coherence_20': array([0.72912617]),
    'diversity_euclidean': 0.04538816307890767,
    'diversity_jensenshannon': 0.04538816307890767,
    'diversity_hellinger': 0.04538816307890767,
    'diversity_cosine': 0.04538816307890767},
   'topic_coherences': {0: 0.4747744148677699,
    1: 0.7036357726509829,
    2: 0.6385750582879668,
    3: 0.8588277998967593,
    4: 0.8910872346278833,
    5: 0.8629812802644291,
    6: 0.6893087965800643,
    7: 0.5541064624279066,
    8: 0.8431573420660817,
    9: 0.922791744160493,
    10: 1.1066082291804975,
    11: 0.4860267452007646,
    12: 0.6664376516550651,
    13: 0.4953589831508357,
    14: 0.4460860341997396,
    15: 0.865571529527662,
    16: 1.0827362012402582,
    17: 0.6510241808664234,
    18: 0.7806403172775794,
    19: 0.5627876636567518}},
  {'scores': {'perplexity': 2964.62841796875,
    'coherence_20': array([0.742071]),
    'diversity_euclidean': 0.04512760284455174,
    

In [44]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

symmetric 2980.0504557291665
asymmetric 2977.8008626302085
heuristic 3156.46337890625


In [ ]:
# Best: asymmetric 2977.8008626302085

In [31]:
BEST_PARAMS[KnownModel.LDA] = {
    'prior': 'asymmetric',
}

In [37]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.LDA: 'LDA'>: {'prior': 'asymmetric'}}

In [48]:
BEST_PARAMS = {KnownModel.PLSA: None,
     KnownModel.DECORRELATION: {'decorrelation_tau': 0.01,
      'sparse_sp_tau': -0.05,
      'smooth_bcg_tau': 0.05},
     KnownModel.TLESS: None,
     KnownModel.SPARSE: {'sparse_sp_tau': -0.05,
      'smooth_bcg_tau': 0.05},
     KnownModel.LDA: {'prior': 'asymmetric'}}

In [49]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.LDA: 'LDA'>: {'prior': 'asymmetric'}}

In [50]:
import json
import warnings

warnings.simplefilter('ignore', UserWarning)

In [47]:
scores.keys()

dict_keys(['scores', 'topic_coherences', 'topic_coherences_toplen_ptw', 'topic_coherences_topden_ptw'])

In [51]:
NUM_TRAINS = 20  # 100
COHERENCES = {
    'topic_coherences': list(),
    # 'topic_coherences_toplen_pwt': list(),
    'topic_coherences_toplen_ptw': list(),
    'topic_coherences_topden_ptw': list(),
}

In [52]:
SAVE_FOLDER = 'results_intra/postnauka'

! mkdir -p $SAVE_FOLDER

In [53]:
! ls

20_Newsgroups__internals
_ARTM-Models-20NewsGroups-T20.ipynb
ARTM-Models-20NewsGroups-T20.ipynb
ARTM-Models-20NewsGroups-T50.ipynb
ARTM-Models-MKB10-T20-Copy1.ipynb
ARTM-Models-MKB10-T20.ipynb
ARTM-Models-MKB10-T50.ipynb
ARTM-Models-PostNauka-T20-Intra.ipynb
ARTM-Models-PostNauka-T20.ipynb
ARTM-Models-PostNauka-T50.ipynb
ARTM-Models-RTL-Wiki-Person-T20-Copy1.ipynb
ARTM-Models-RTL-Wiki-Person-T20.ipynb
ARTM-Models-RTL-Wiki-Person-T50-Copy1.ipynb
ARTM-Models-RTL-Wiki-Person-T50.ipynb
ARTM-Models-RuWikiGood-T20.ipynb
ARTM-Models-RuWikiGood-T50.ipynb
BERTopic
BERTopic-Coherence-20NewsGroups-T20.ipynb
BERTopic-Coherence-20NewsGroups-T50.ipynb
BERTopic-Coherence-MKB10-T20.ipynb
BERTopic-Coherence-MKB10-T50.ipynb
BERTopic-Coherence-PostNauka-T20.ipynb
BERTopic-Coherence-PostNauka-T50.ipynb
BERTopic-Coherence-RTL-Wiki-Person-T20.ipynb
BERTopic-Coherence-RTL-Wiki-Person-T50.ipynb
BERTopic-Coherence-RuWikiGood-T20.ipynb
BERTopic-Coherence-RuWikiGood-T50-2.ipynb
_BERTopic-Coherence-RuWikiGood-T50

In [54]:
! ls $SAVE_FOLDER

plsa_with_cohs.json


In [55]:
# PLSA

start = time.time()

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.PLSA,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
    )
    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    # COHERENCES.extend(
    #     list(results['topic_coherences'].values())
    # )

    for k in COHERENCES:
        COHERENCES[k].extend(
            list(results[k].values())
        )

end = time.time()

print(f'Elapsed: {(end - start) / 60} min')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 76.59856515725454 min


In [56]:
1

1

In [81]:
scores[-1]

{'scores': {'perplexity': 2481.011474609375,
  'coherence_20': array([0.69311395]),
  'toplen_pwt': 2.2419136748104247,
  'toplen_ptw': 2.2419136748104247,
  'topden_ptw': 0.7597696858931959,
  'diversity_euclidean': 0.051507117819155505,
  'diversity_jensenshannon': 0.5886639008885491,
  'diversity_hellinger': 0.6731241634368506,
  'diversity_cosine': 0.716436824471558},
 'topic_coherences': {0: 0.5133376224986828,
  1: 0.5636795412880685,
  2: 0.734942197901494,
  3: 1.106716850102659,
  4: 0.5436879375765777,
  5: 0.5289428294650398,
  6: 0.6085041139511407,
  7: 0.4131289730452274,
  8: 0.7236630293837298,
  9: 0.5207321747706948,
  10: 0.7016884077306269,
  11: 0.8370214440333109,
  12: 0.5981763032489854,
  13: 0.7666095142248066,
  14: 0.40974011652763215,
  15: 0.7858678342478069,
  16: 0.8803072445751484,
  17: 1.240285463956784,
  18: 0.8492280572719985,
  19: 0.5360192650721023},
 'topic_coherences_toplen_pwt': {0: 2.251163602635274,
  1: 2.5048643275639324,
  2: 2.337196485

In [82]:
scores

[{'scores': {'perplexity': 2474.488525390625,
   'coherence_20': array([0.73062234]),
   'toplen_pwt': 2.2222272562846417,
   'toplen_ptw': 2.2222272562846417,
   'topden_ptw': 0.7928443137523,
   'diversity_euclidean': 0.05432938655690007,
   'diversity_jensenshannon': 0.6000053864026371,
   'diversity_hellinger': 0.6857673790274449,
   'diversity_cosine': 0.7377318963938092},
  'topic_coherences': {0: 0.4682133200037822,
   1: 0.5525499039748306,
   2: 0.3662412591097539,
   3: 0.7660746688638103,
   4: 0.6599584504901529,
   5: 1.2151843412918593,
   6: 0.784374938331295,
   7: 0.6583524284174844,
   8: 0.7558873960285204,
   9: 0.8872288724338003,
   10: 0.5060579830596448,
   11: 1.1643195534386466,
   12: 1.0551562205101295,
   13: 0.6667121824386473,
   14: 0.34910355580365166,
   15: 0.7823822821267535,
   16: 1.0022462210267367,
   17: 0.8400496475709041,
   18: 0.706545516531718,
   19: 0.425808078140945},
  'topic_coherences_toplen_pwt': {0: 2.0956432149414908,
   1: 2.15352

In [57]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [58]:
scores[0]

{'scores': {'perplexity': 2474.488525390625,
  'coherence_20': 0.7306223409796534,
  'toplen_ptw': 2.2222272562846417,
  'topden_ptw': 0.4862072599670759,
  'diversity_euclidean': 0.05432938184650551,
  'diversity_jensenshannon': 0.6000053797380491,
  'diversity_hellinger': 0.6857673609277269,
  'diversity_cosine': 0.7377318746693401},
 'topic_coherences': {0: 0.4682133200037822,
  1: 0.5525499039748306,
  2: 0.3662412591097539,
  3: 0.7660746688638103,
  4: 0.6599584504901529,
  5: 1.2151843412918593,
  6: 0.784374938331295,
  7: 0.6583524284174844,
  8: 0.7558873960285204,
  9: 0.8872288724338003,
  10: 0.5060579830596448,
  11: 1.1643195534386466,
  12: 1.0551562205101295,
  13: 0.6667121824386473,
  14: 0.34910355580365166,
  15: 0.7823822821267535,
  16: 1.0022462210267367,
  17: 0.8400496475709041,
  18: 0.706545516531718,
  19: 0.425808078140945},
 'topic_coherences_toplen_ptw': {0: 2.0956432149414908,
  1: 2.1535240947120737,
  2: 2.8292371579349185,
  3: 1.87571387898929,
  4:

In [59]:
with open(SAVE_FOLDER + '/plsa_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [60]:
COHERENCES

{'topic_coherences': [0.4682133200037822,
  0.5525499039748306,
  0.3662412591097539,
  0.7660746688638103,
  0.6599584504901529,
  1.2151843412918593,
  0.784374938331295,
  0.6583524284174844,
  0.7558873960285204,
  0.8872288724338003,
  0.5060579830596448,
  1.1643195534386466,
  1.0551562205101295,
  0.6667121824386473,
  0.34910355580365166,
  0.7823822821267535,
  1.0022462210267367,
  0.8400496475709041,
  0.706545516531718,
  0.425808078140945,
  1.3242652199570513,
  0.5722318709679326,
  0.5943743640997695,
  0.690679616728474,
  0.539257289894177,
  0.37097788905119117,
  0.4497302529498234,
  0.7614405692714401,
  0.5175776821765838,
  0.6621937385963241,
  0.8423462341482593,
  0.708637600018853,
  0.41846542738493603,
  0.6254327699417653,
  1.213484947540808,
  0.8842201698926271,
  0.7557284244360403,
  0.6408219757743792,
  0.511304302409602,
  0.730423536324743,
  0.5020493941601943,
  0.5664712103832005,
  0.7641518298953729,
  0.46302107182953806,
  0.4595996841453

In [61]:
len(COHERENCES['topic_coherences'])

400

In [89]:
COHERENCES['topic_coherences'][:10]

[0.4682133200037822,
 0.5525499039748306,
 0.3662412591097539,
 0.7660746688638103,
 0.6599584504901529,
 1.2151843412918593,
 0.784374938331295,
 0.6583524284174844,
 0.7558873960285204,
 0.8872288724338003]

In [90]:
COHERENCES['topic_coherences'][:20]

[0.4682133200037822,
 0.5525499039748306,
 0.3662412591097539,
 0.7660746688638103,
 0.6599584504901529,
 1.2151843412918593,
 0.784374938331295,
 0.6583524284174844,
 0.7558873960285204,
 0.8872288724338003,
 0.5060579830596448,
 1.1643195534386466,
 1.0551562205101295,
 0.6667121824386473,
 0.34910355580365166,
 0.7823822821267535,
 1.0022462210267367,
 0.8400496475709041,
 0.706545516531718,
 0.425808078140945]

In [62]:
# Sparse

start = time.time()

scores = []

# for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
for seed in range(NUM_TRAINS):
    if seed != NUM_TRAINS - 1:
        print(seed, end=' ')
    else:
        print(seed)

    model = init_model_from_family(
        family=KnownModel.SPARSE,
        dataset=dataset,
        main_modality=MAIN_MODALITY,
        num_topics=NUM_TOPICS,
        seed=seed,
        model_params=BEST_PARAMS[KnownModel.SPARSE],
    )

    if seed == 0:
        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

    results = fit_and_compute_scores(model, dataset)
    # scores.append(results['scores'])
    scores.append(results)

    # COHERENCES.extend(
    #     list(results['topic_coherences'].values())
    # )

    for k in COHERENCES:
        COHERENCES[k].extend(
            list(results[k].values())
        )

end = time.time()

print(f'Elapsed: {(end - start) / 60} min')

0 smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
sparse_phi_sp: -0.24148051194680664
sparse_theta_sp: -1.3690669651493796
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 83.63650202353796 min


In [63]:
# for s in scores:
#     s['coherence_20'] = float(s['coherence_20'])

for s in scores:
    s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [64]:
with open(SAVE_FOLDER + '/sparse_with_cohs.json', 'w') as f:
    f.write(
        json.dumps(scores, indent=4)
    )

In [63]:
len(COHERENCES)

800

In [64]:
COHERENCES[-10:]

[0.880095624600911,
 0.9270009959013572,
 0.452872601334081,
 0.9753726382102174,
 0.4825404955153426,
 0.7157639092437795,
 0.632902255675745,
 0.7792618758677884,
 0.8436468554657421,
 0.6953114019848466]

In [65]:
max(COHERENCES)

1.3530270785302947

In [65]:
def train_many(model_family, save_file_path):
    start = time.time()
    
    scores = []

    # for seed in tqdm(range(NUM_TRAINS), total=NUM_TRAINS):
    for seed in range(NUM_TRAINS):
        if seed != NUM_TRAINS - 1:
            print(seed, end=' ')
        else:
            print(seed)
    
        model = init_model_from_family(
            family=model_family,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params=BEST_PARAMS[model_family],
        )
    
        if seed == 0:
            for reg in model.regularizers.data:
                print(f"{reg}: {model.regularizers[reg].tau}")
    
        results = fit_and_compute_scores(model, dataset)
        # scores.append(results['scores'])
        scores.append(results)
    
        # COHERENCES.extend(
        #     list(results['topic_coherences'].values())
        # )
    
        for k in COHERENCES:
            COHERENCES[k].extend(
                list(results[k].values())
            )
    
    end = time.time()
    
    print(f'Elapsed: {(end - start) / 60} min')

    # for s in scores:
    #     s['coherence_20'] = float(s['coherence_20'])
    
    for s in scores:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

    with open(save_file_path, 'w') as f:
        f.write(
            json.dumps(scores, indent=4)
        )

In [ ]:
# well, we missed it... whatever
# train_many(KnownModel.DECORRELATION, SAVE_FOLDER + '/decorrelation_with_cohs.json')

In [66]:
train_many(KnownModel.DECORRELATION, SAVE_FOLDER + '/decorrelation_with_cohs.json')

0 decorrelation: 0.01
smooth_phi_bcg: 5.337990264087306
smooth_theta_bcg: 30.263585545407338
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 80.19879750410716 min


In [68]:
len(COHERENCES)

1200

In [67]:
train_many(KnownModel.TLESS, SAVE_FOLDER + '/tless_with_cohs.json')

0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 82.0886171221733 min


In [72]:
len(COHERENCES)

1600

In [68]:
train_many(KnownModel.LDA, SAVE_FOLDER + '/lda_with_cohs.json')

0 smooth_phi: 0.05
smooth_theta_0: 0.12456869333982468
smooth_theta_1: 0.10180450975894928
smooth_theta_2: 0.08607485145330429
smooth_theta_3: 0.0745554119348526
smooth_theta_4: 0.06575533747673035
smooth_theta_5: 0.05881335958838463
smooth_theta_6: 0.053197186440229416
smooth_theta_7: 0.04856010526418686
smooth_theta_8: 0.04466661810874939
smooth_theta_9: 0.041351135820150375
smooth_theta_10: 0.038493841886520386
smooth_theta_11: 0.036005899310112
smooth_theta_12: 0.033820029348134995
smooth_theta_13: 0.03188437223434448
smooth_theta_14: 0.03015829436480999
smooth_theta_15: 0.02860950119793415
smooth_theta_16: 0.02721201814711094
smooth_theta_17: 0.025944700464606285
smooth_theta_18: 0.024790173396468163
smooth_theta_19: 0.023734018206596375
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
Elapsed: 80.19886320829391 min


In [41]:
len(COHERENCES)

2000

In [54]:
1

1

In [71]:
COHERENCES.keys()

dict_keys(['topic_coherences', 'topic_coherences_toplen_ptw', 'topic_coherences_topden_ptw'])

In [72]:
for k in COHERENCES:
    print(k)
    print(len(COHERENCES[k]))

topic_coherences
2000
topic_coherences_toplen_ptw
2000
topic_coherences_topden_ptw
2000


In [77]:
for p in range(5, 100, 5):
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 5: 0.44909245458611957
10: 0.49633189795078303
15: 0.5312798544554967
20: 0.5568860704432789
25: 0.583641029664594
30: 0.6154803648381737
35: 0.6420080477800698
40: 0.6686072625177087
45: 0.6987521704785715
50: 0.7275350316751499
55: 0.7582740845556143
60: 0.7879524550264069
65: 0.8206503482616105
70: 0.8533797049735502
75: 0.8848794056365471
80: 0.9154723765333351
85: 0.9536494799334573
90: 1.0014147388051453
95: 1.089784010979277


In [74]:
for k in COHERENCES:
    print(k)

    for p in range(5, 100, 5):
        print(f'{p:2}: {np.percentile(COHERENCES[k], p)}')

    print()

topic_coherences
 5: 0.3761569938166719
10: 0.4143367527969667
15: 0.4542455875910814
20: 0.4834872457111293
25: 0.513590898996654
30: 0.5425365977325278
35: 0.565671908329175
40: 0.5961392562354773
45: 0.6277063043052796
50: 0.6607445968186382
55: 0.6904814875133726
60: 0.7210395226390767
65: 0.7526029448855149
70: 0.7852508343981067
75: 0.8319217345735135
80: 0.8825231364747546
85: 0.9321277425861927
90: 1.0118762991083503
95: 1.1294498709977603

topic_coherences_toplen_ptw
 5: 1.7428335802764752
10: 1.7938763485672071
15: 1.8444298959987273
20: 1.890894097939173
25: 1.9310106265475007
30: 1.9751818974915076
35: 2.018198141521423
40: 2.050908263530298
45: 2.0911345123527316
50: 2.13460198747896
55: 2.1653136419406804
60: 2.21950006446868
65: 2.273771732539559
70: 2.3394925682940233
75: 2.435017475164776
80: 2.5432101128006273
85: 2.6917629341488447
90: 2.939062174058066
95: 3.2270515940633073

topic_coherences_topden_ptw
 5: 0.42461509010515663
10: 0.4353252849916843
15: 0.4446734577

In [78]:
min(COHERENCES), max(COHERENCES)

(0.23028683424984975, 1.3530270785302947)

In [80]:
np.argmin(COHERENCES), np.argmax(COHERENCES)

(939, 520)

In [75]:
for k in COHERENCES:
    print(k)
    print(min(COHERENCES[k]), max(COHERENCES[k]))
    print()

topic_coherences
0.15132387398111333 1.5020738514744776

topic_coherences_toplen_ptw
1.5032401948154457 5.126959948774844

topic_coherences_topden_ptw
0.3826427537899631 0.9706315338779786



In [83]:
for p in [2, 98]:
    print(f'{p:2}: {np.percentile(COHERENCES, p)}')

 2: 0.38638722557933053
98: 1.177163998303747


In [76]:
for k in COHERENCES:
    print(k)

    for p in [2, 98]:
        print(f'{p:2}: {np.percentile(COHERENCES[k], p)}')

    print()

topic_coherences
 2: 0.34199022361156417
98: 1.226462260611979

topic_coherences_toplen_ptw
 2: 1.663891658164501
98: 3.604047548237196

topic_coherences_topden_ptw
 2: 0.4138428867647952
98: 0.8520687916766032



In [84]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.LDA: 'LDA'>: {'prior': 'asymmetric'}}

In [77]:
BEST_PARAMS

{<KnownModel.PLSA: 'PLSA'>: None,
 <KnownModel.DECORRELATION: 'decorrelation'>: {'decorrelation_tau': 0.01,
  'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.TLESS: 'TARTM'>: None,
 <KnownModel.SPARSE: 'sparse'>: {'sparse_sp_tau': -0.05,
  'smooth_bcg_tau': 0.05},
 <KnownModel.LDA: 'LDA'>: {'prior': 'asymmetric'}}

In [78]:
COHERENCES

{'topic_coherences': [0.4682133200037822,
  0.5525499039748306,
  0.3662412591097539,
  0.7660746688638103,
  0.6599584504901529,
  1.2151843412918593,
  0.784374938331295,
  0.6583524284174844,
  0.7558873960285204,
  0.8872288724338003,
  0.5060579830596448,
  1.1643195534386466,
  1.0551562205101295,
  0.6667121824386473,
  0.34910355580365166,
  0.7823822821267535,
  1.0022462210267367,
  0.8400496475709041,
  0.706545516531718,
  0.425808078140945,
  1.3242652199570513,
  0.5722318709679326,
  0.5943743640997695,
  0.690679616728474,
  0.539257289894177,
  0.37097788905119117,
  0.4497302529498234,
  0.7614405692714401,
  0.5175776821765838,
  0.6621937385963241,
  0.8423462341482593,
  0.708637600018853,
  0.41846542738493603,
  0.6254327699417653,
  1.213484947540808,
  0.8842201698926271,
  0.7557284244360403,
  0.6408219757743792,
  0.511304302409602,
  0.730423536324743,
  0.5020493941601943,
  0.5664712103832005,
  0.7641518298953729,
  0.46302107182953806,
  0.4595996841453

## Newman Vs. Intratext

### Correlation

In [81]:
COHERENCES.keys()

dict_keys(['topic_coherences', 'topic_coherences_toplen_ptw', 'topic_coherences_topden_ptw'])

In [79]:
from scipy.stats import spearmanr

In [82]:
spearmanr(
    COHERENCES['topic_coherences'],
    COHERENCES['topic_coherences_toplen_ptw'],
)

SignificanceResult(statistic=0.25355294374651943, pvalue=1.0350547711474095e-30)

In [83]:
spearmanr(
    COHERENCES['topic_coherences'],
    COHERENCES['topic_coherences_topden_ptw'],
)

SignificanceResult(statistic=0.1523687495268863, pvalue=7.389691451133996e-12)

In [84]:
spearmanr(
    COHERENCES['topic_coherences_toplen_ptw'],
    COHERENCES['topic_coherences_topden_ptw'],
)

SignificanceResult(statistic=0.0064989661247415315, pvalue=0.7714613154312437)

### Tops Intersection

In [87]:
len(COHERENCES['topic_coherences'])

2000

In [88]:
inds1 = np.argsort(COHERENCES['topic_coherences'])[::-1][:100]

In [89]:
inds1

array([ 411,  288,  910,  761, 1888, 1088,   68, 1668, 1556, 1161,  420,
         20, 1319,  985, 1383, 1961,  361,  810,  678,  345, 1945,  468,
        313,  525,  434,  972, 1145, 1235,  902,  405,  834,  610, 1913,
        497,   97,  820, 1153,  805, 1710,  110,  578,  978, 1354, 1278,
       1697,    5, 1605,   34, 1634, 1620,  550,  453,  589,  897, 1515,
       1216,  668, 1068, 1115,  983, 1296,  416,  457,  713,  687,  925,
        688,  791,  510,  989,  125, 1725,  189, 1789, 1191,   11, 1444,
       1236,  172,  319, 1919, 1035,  541,  868,  937,  210, 1810,  278,
       1878, 1485, 1107, 1584,  857, 1737, 1248, 1437, 1519, 1010, 1060,
       1361])

In [92]:
COHERENCES['topic_coherences'][inds1[0]], COHERENCES['topic_coherences'][inds1[1]]

(1.5020738514744776, 1.4401240021677997)

In [94]:
inds1 = set(np.argsort(COHERENCES['topic_coherences'])[::-1][:100])
inds2 = set(np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:100])
inds3 = set(np.argsort(COHERENCES['topic_coherences_topden_ptw'])[::-1][:100])

In [97]:
len(inds1 & inds2), len(inds1 & inds3), len(inds2 & inds3)

(31, 16, 36)

In [98]:
inds1 = set(np.argsort(COHERENCES['topic_coherences'])[::-1][:200])
inds2 = set(np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:200])
inds3 = set(np.argsort(COHERENCES['topic_coherences_topden_ptw'])[::-1][:200])

In [99]:
len(inds1 & inds2), len(inds1 & inds3), len(inds2 & inds3)

(97, 38, 51)

In [100]:
inds1 = set(np.argsort(COHERENCES['topic_coherences'])[::-1][:500])
inds2 = set(np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:500])
inds3 = set(np.argsort(COHERENCES['topic_coherences_topden_ptw'])[::-1][:500])

In [101]:
len(inds1 & inds2), len(inds1 & inds3), len(inds2 & inds3)

(258, 149, 142)

### Newman-in-Intratext Density (Mutual Density / Intra-Top Density)

In [109]:
inds2 = np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:100]
inds3 = np.argsort(COHERENCES['topic_coherences_topden_ptw'])[::-1][:100]

In [110]:
np.mean(
    np.array(COHERENCES['topic_coherences'])
)

0.6880267784698945

In [111]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds2]
)

0.9929759853960232

In [112]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds3]
)

0.9230413598218548

In [113]:
inds2 = np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:200]
inds3 = np.argsort(COHERENCES['topic_coherences_topden_ptw'])[::-1][:200]

In [114]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds2]
)

0.9272760323965406

In [115]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds3]
)

0.7842457416235419

In [116]:
inds2 = np.argsort(COHERENCES['topic_coherences_toplen_ptw'])[::-1][:500]
inds3 = np.argsort(COHERENCES['topic_coherences_topden_ptw'])[::-1][:500]

In [117]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds2]
)

0.8172972792504491

In [118]:
np.mean(
    np.array(COHERENCES['topic_coherences'])[inds3]
)

0.6594688636842502